# Case study: Naïve Bayes on the Amazon reviews
**Web and Text Analytics 2026-27 · Week 6 · Learning weights, learning words**

In Week 5 we trained a logistic regression on the Amazon reviews. Here we train a Naïve Bayes model on **the same reviews, the same split and the same document-term matrix**.
Only the model changes, so any difference in the results comes from the model.

Each section below follows the case study slides, in the same order and with the same numbers.

Data: https://www.kaggle.com/nehasontakke/amazon-unlocked-mobilecsv · two ways to load it, see **Load the data** below.
Checked with scikit-learn 1.9.1, pandas 3.0 and numpy 2.5.

## Load the data
Choose one of the two options with `SOURCE`:

- **`"upload"`**: you have `Amazon_Unlocked_Mobile.csv` on your computer. In Colab, upload it from the **Files** panel on the left
  (or uncomment the two `files.upload()` lines). On your own computer, put it in the same folder as this notebook.
- **`"kaggle"`**: the notebook downloads the dataset from Kaggle with `kagglehub`, no upload needed (about 130 MB, it takes a minute).

In [ ]:
SOURCE = "kaggle"   # "upload" or "kaggle"

if SOURCE == "kaggle":
    import glob, os
    import kagglehub   # in Colab already installed; elsewhere: pip install kagglehub
    folder = kagglehub.dataset_download("nehasontakke/amazon-unlocked-mobilecsv")
    csv_path = glob.glob(os.path.join(folder, "*.csv"))[0]   # the CSV file of the dataset
else:
    # from google.colab import files
    # files.upload()   # choose Amazon_Unlocked_Mobile.csv
    csv_path = "Amazon_Unlocked_Mobile.csv"

print(csv_path)

## Setup · the data of Week 5
The first steps of the Week 5 notebook, in one cell: 10% sample, no missing values, no 3-star reviews, `Positively Rated` (1 for 4–5 stars, 0 for 1–2), the same train–test split
and the same `CountVectorizer`, fitted on `X_train` only.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

df = pd.read_csv(csv_path)   # 413,840 reviews
df = df.sample(frac=0.1, random_state=10)   # 10%: 41,384 reviews
df.dropna(inplace=True)   # drop missing values
df = df[df["Rating"] != 3]   # remove neutral ratings
df["Positively Rated"] = np.where(df["Rating"] > 3, 1, 0)

X_train, X_test, y_train, y_test = train_test_split(df["Reviews"],
                                                    df["Positively Rated"],
                                                    random_state=0)

vect = CountVectorizer().fit(X_train)   # vocabulary from the training reviews only
X_train_vectorized = vect.transform(X_train)   # document-term matrix
feature_names = np.array(vect.get_feature_names_out())

print("Training reviews:", X_train.shape[0])   # 23,052
print("Test reviews:", X_test.shape[0])   # 7,684
print("Vocabulary:", len(feature_names))   # 19,533

## Step 0 · Collect and annotate the corpus
The corpus is the training set. The annotation came for free: the star rating.

In [ ]:
y = y_train.values   # 1 = positive, 0 = negative
D_pos = (y == 1).sum()
D_neg = (y == 0).sum()
print("Positive reviews:", D_pos)   # 17,214
print("Negative reviews:", D_neg)   # 5,838

In [ ]:
print(X_train[X_train.str.startswith("Great phone for the price")].iloc[0])
print(X_train[X_train.str.startswith("The phone did not work")].iloc[0])

## Step 1 · Preprocess
As in Week 5, `CountVectorizer` does this step for us: lowercase, remove punctuation, tokenize (words of two or more letters).
`build_analyzer()` shows what it does to a single review.

In [ ]:
analyzer = vect.build_analyzer()
print(analyzer("Great phone for the price."))   # ['great', 'phone', 'for', 'the', 'price']
print(analyzer("The phone did not work"))   # ['the', 'phone', 'did', 'not', 'work']

## Step 2 · Word count
`freq(w, class)`: how many times each word appears in all the reviews of the class.
It is the document-term matrix of Week 5 with its rows added up per class: one sum for the positive rows, one for the negative rows.

In [ ]:
freq_pos = np.asarray(X_train_vectorized[y == 1].sum(axis=0)).ravel()   # one number per word
freq_neg = np.asarray(X_train_vectorized[y == 0].sum(axis=0)).ravel()

words = ["great", "good", "not", "the", "phone", "is", "defective"]
idx = [vect.vocabulary_[w] for w in words]   # column of each word in the matrix

pd.DataFrame({"freq(w, pos)": freq_pos[idx], "freq(w, neg)": freq_neg[idx]}, index=words)

Raw counts are not comparable yet: the positive class has about twice as many words.

In [ ]:
N_pos = freq_pos.sum()
N_neg = freq_neg.sum()
print("Words in positive reviews, N_pos:", N_pos)   # 584,330
print("Words in negative reviews, N_neg:", N_neg)   # 278,648

## Step 3 · P(w | class)
Divide each count by the number of words in its class, with **Laplacian smoothing**: add 1 to every count and V to every denominator,
so that no word gets a probability of zero.

$$P(w \mid class) = \frac{freq(w, class) + 1}{N_{class} + V}$$

In [ ]:
V = len(feature_names)
print("Denominators:", N_pos + V, N_neg + V)   # 603,863 and 298,181

p_pos = (freq_pos + 1) / (N_pos + V)
p_neg = (freq_neg + 1) / (N_neg + V)

pd.DataFrame({"P(w | pos)": p_pos[idx], "P(w | neg)": p_neg[idx]}, index=words).round(5)

In [ ]:
print(p_pos.sum(), p_neg.sum())   # each column of probabilities adds up to 1

## Step 4 · Get lambda
$$\lambda(w) = \log \frac{P(w \mid pos)}{P(w \mid neg)}$$

Above 0: positive word. Below 0: negative word. Close to 0: neutral.

In [ ]:
lam = np.log(p_pos / p_neg)   # one lambda per word: 19,533 numbers

pd.DataFrame({"P(w | pos)": p_pos[idx], "P(w | neg)": p_neg[idx], "lambda": lam[idx]},
             index=words).round({"P(w | pos)": 5, "P(w | neg)": 5, "lambda": 2})   # lambda(great) = 1.82

### Smallest and largest lambdas
The words that Naïve Bayes counts as most negative and most positive. Compare them with the coefficients of logistic regression in Week 5.

In [ ]:
sorted_lam_index = lam.argsort()
print("Smallest lambdas:", feature_names[sorted_lam_index[:10]])
print("Largest lambdas:", feature_names[sorted_lam_index[:-11:-1]])

## Step 5 · Get the log prior
$$logprior = \log \frac{D_{pos}}{D_{neg}}$$

About three positive reviews for each negative one: every review starts from +1.08 before its words are read.

In [ ]:
logprior = np.log(D_pos / D_neg)
print("log prior:", round(logprior, 2))   # 1.08

## Predict using Naïve Bayes
score = logprior + the sum of the lambdas of the words in the review. A word that appears twice counts twice.

Multiplying the document-term matrix by the vector of lambdas does this sum for every review at once.

In [ ]:
def predict(reviews):
    """Naïve Bayes score of each review: logprior + sum of lambdas."""
    return logprior + vect.transform(reviews) @ lam

review = "The phone is defective"
for w in analyzer(review):
    print(w, round(lam[vect.vocabulary_[w]], 2))   # the -0.03, phone -0.01, is 0.28, defective -1.99

score = predict([review])[0]
print("score:", round(score, 2))   # -0.67: negative

As in Part I, the sigmoid turns the score into a probability.

In [ ]:
print("P(positive):", round(1 / (1 + np.exp(-score)), 2))   # 0.34

## Test Naïve Bayes
The 7,684 test reviews of Week 5 were set aside during training. We compute the score of each one; a score above 0 means positive.

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score

scores = predict(X_test)
pred = (scores > 0).astype(int)
print(scores[:4].round(2))   # 2.15, 4.79, 10.44, -6.52
print(pred[:4])   # 1, 1, 1, 0

In [ ]:
print("Accuracy:", round(accuracy_score(y_test, pred), 3))   # 0.916
print("Always positive:", round(y_test.mean(), 3))   # 0.748
print("AUC:", round(roc_auc_score(y_test, scores), 3))   # 0.948